<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

# 📚 Recommender Systems: From "People Also Bought" to Deep Learning
### CP020003 Artificial Intelligence — In-Class Notebook (Recommender Systems Deep-Dive)

Last week you discovered *hidden groups* in customers with no label at all (unsupervised learning). This week
we tackle a problem almost every company with more than 100 products eventually faces: **"Of the thousands of
items we have, which few should we show *this* user, right now?"** That is the job of a **Recommender System**
— arguably the single most economically valuable application of machine learning on the internet today
(Amazon, Netflix, Spotify, YouTube, TikTok all run on some flavor of it).

We will use the real **Book-Crossing dataset** — over a million book ratings from ~280,000 readers on
~271,000 books — and build our way up from the simplest possible recommender (just show what's popular) all
the way to a **deep learning** recommender with learned user/item embeddings, exactly the same core idea that
powers YouTube's and TikTok's "For You" feeds.

By the end of this notebook you will be able to:

1. Explain the recommender-systems **family tree**: content-based, collaborative filtering (user-based,
   item-based, model-based), hybrid, and deep-learning-based approaches
2. Explain **why raw rating matrices are almost always extremely sparse**, and why that matters
3. Build a **Popularity-Based** recommender — the "dumb but strong" cold-start baseline every system needs
4. Build **User-Based** and **Item-Based Collaborative Filtering** with cosine similarity, and explain why
   item-based tends to win in production
5. Build a **Matrix Factorization** recommender (SVD) — the technique that won the famous $1M Netflix Prize
6. Build a **Content-Based** recommender using TF-IDF on book titles/authors — the fix for the "cold-start
   item" problem
7. Build a small **Neural Collaborative Filtering (NCF)** model with `Keras` — the deep-learning version of
   matrix factorization
8. Evaluate recommenders properly with **RMSE**, **Precision@K**, **Recall@K**, and **Coverage** — not just
   accuracy
9. Know the essential **tricks and pitfalls**: cold start, popularity bias, implicit vs. explicit feedback,
   negative sampling, and evaluation leakage
10. Get a guided tour of **modern, production-grade techniques**: two-tower retrieval, sequential/session-based
    transformers, graph neural networks, and LLM-based recommenders
11. Turn a recommender's output into a **business action**, not just a list of ISBNs

Runs fully on **Google Colab (free CPU)** — the deep-learning section trains in well under a minute, no GPU
required. ⏱️


## 0. Setup

We need the usual data-science stack (`pandas`, `numpy`, `matplotlib`, `seaborn`), `scikit-learn` for
similarity/SVD/TF-IDF/metrics, and `tensorflow`/`keras` for the deep-learning section. Colab already ships
with all of these pre-installed, so this cell just imports them.


In [ ]:
# Run this once per Colab session if a package is ever missing
# !pip -q install scikit-learn tensorflow --upgrade

import warnings
warnings.filterwarnings("ignore")

import io
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

print("Libraries loaded \u2705")

In [ ]:
RANDOM_STATE = # Pick your lucky number here
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

## 1. Why Recommender Systems? 🤔

| | **Search** | **Recommendation** |
|---|---|---|
| **User intent** | Explicit — the user typed a query | Implicit — the user just wants *something good*, and may not know what |
| **Question it answers** | *"Show me items matching 'blue running shoes'"* | *"Show me items **I** would like, that I didn't even know to search for"* |
| **Core input** | A query string | The user's **history** + everyone else's history |
| **Business examples** | Google Search, site search bar | Netflix "Because you watched...", Amazon "Customers also bought", Spotify Discover Weekly |

### The catalog problem 🏬

A bookstore with 10 books on a shelf doesn't need AI — a human can browse all of them. Our dataset has
**271,000 books**. No reader will ever browse that shelf. Every platform with a catalog too large to browse
manually eventually needs the same tool: something that narrows *millions* of options down to a handful that
matter for *this specific person*.

That's the whole point of a recommender system: **personalized filtering at scale.**


## 2. The Recommender Systems Family Tree 🌳

There are dozens of named algorithms out there, but almost all of them fall into one of these buckets:

| Family | Core idea | Needs... | Classic weakness |
|---|---|---|---|
| **Popularity-based** | Recommend what's popular / highly rated overall | Nothing personal at all | Same list for everyone — zero personalization |
| **Content-Based Filtering** | *"You liked items with these features → here are more items with similar features"* | Item metadata (title, genre, author, text...) | Over-specializes — keeps recommending the same thing |
| **Collaborative Filtering — User-Based** | *"Users similar to you liked X"* | A user-item rating matrix | Doesn't scale — comparing millions of users is expensive |
| **Collaborative Filtering — Item-Based** | *"You liked X → here are items similar **users** also liked alongside X"* | A user-item rating matrix | Still struggles with brand-new items (cold start) |
| **Model-Based CF (Matrix Factorization)** | Learn hidden **latent factors** for every user and item (e.g. via SVD) | A user-item rating matrix | Latent factors aren't human-interpretable |
| **Deep-Learning-Based (Neural CF, Two-Tower, Transformers)** | Learn user/item **embeddings** with a neural network instead of linear algebra | Lots of data, some compute | Needs more data & tuning than classical methods to pay off |
| **Hybrid** | Combine two or more of the above (e.g. content-based for new items + CF for everything else) | Whatever its components need | More moving parts to maintain |

We will build **one working example of almost every row in this table** today, on the same dataset, so you can
directly compare them.


## 3. Load the Dataset 📥

**Dataset: Book-Crossing** — real-world book ratings collected from the Book-Crossing community.

* `Books.csv` — 271,360 books (ISBN, title, author, year, publisher, cover image URLs)
* `Ratings.csv` — 1,149,780 ratings (`User-ID`, `ISBN`, `Book-Rating` from **0–10**, where **0 means an
  *implicit* interaction** — the user looked at / owns the book but left no explicit star rating — and 1–10 is
  an *explicit* rating)
* `Users.csv` — 278,858 users (`User-ID`, `Location`, `Age`)

**Credit:** Originally collected by Cai-Nicolas Ziegler (2004). We load our class copy directly from GitHub so
everyone works with the exact same file.

We wrap the download in a `try/except` — if the network call ever fails (e.g. a flaky Colab session), we fall
back to a small embedded sample so the rest of the notebook still runs end-to-end.


In [ ]:
YOUR_DATASET_ZIP_NAME = " # Write your dataset name (.zip) here "
DATA_URL = f"https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/raw/main/dataset/{YOUR_DATASET_ZIP_NAME}"

def load_book_crossing():
    try:
        r = requests.get(DATA_URL, timeout=30)
        r.raise_for_status()
        z = zipfile.ZipFile(io.BytesIO(r.content))
        # files live inside a top-level "book/" folder in the zip
        names = {n.split("/")[-1]: n for n in z.namelist() if n.lower().endswith(".csv")}

        def read(fname):
            with z.open(names[fname]) as f:
                return pd.read_csv(f, encoding="latin-1", on_bad_lines="skip", low_memory=False)

        books = read("Books.csv")
        ratings = read("Ratings.csv")
        users = read("Users.csv")
        print("Loaded from GitHub \u2705")
        return books, ratings, users, True
    except Exception as e:
        print(f"Download failed ({e}). Falling back to a small embedded sample so the notebook still runs.")
        books = pd.DataFrame({
            "ISBN": ["0195153448", "0002005018", "0060973129", "0374157065", "0399135782"],
            "Book-Title": ["Classical Mythology", "Clara Callan", "Decision in Normandy",
                            "Flu", "The Kitchen God's Wife"],
            "Book-Author": ["Mark P. O. Morford", "Richard Bruce Wright", "Carlo D'Este",
                             "Gina Bari Kolata", "Amy Tan"],
            "Year-Of-Publication": [2002, 2001, 1991, 1999, 1991],
            "Publisher": ["Oxford University Press", "HarperFlamingo Canada", "HarperPerennial",
                          "Farrar Straus Giroux", "Putnam Pub Group"],
        })
        ratings = pd.DataFrame({
            "User-ID": [276725, 276726, 276727, 276729, 276729, 276736, 276744],
            "ISBN": ["0195153448", "0002005018", "0060973129", "0374157065", "0399135782",
                     "0002005018", "0195153448"],
            "Book-Rating": [0, 5, 0, 6, 8, 7, 9],
        })
        users = pd.DataFrame({"User-ID": [276725, 276726, 276727, 276729, 276736, 276744],
                               "Location": ["nyc, new york, usa"] * 6,
                               "Age": [25, 30, np.nan, 22, 40, np.nan]})
        return books, ratings, users, False

books, ratings, users, is_full_data = load_book_crossing()
print(f"books={books.shape}  ratings={ratings.shape}  users={users.shape}")


## 4. First Look & EDA 🔍

Before building anything, look at the shape of the problem. Two questions matter most for a recommender:

1. **How are ratings distributed?** (Are most interactions implicit "0"s, or real star ratings?)
2. **How sparse is the user-item matrix?** (This single number decides which algorithms will even work.)


In [ ]:
print(ratings["Book-Rating"].describe())
print()
print("Share of ratings that are 0 (implicit / no explicit score):",
      f"{(ratings['Book-Rating'] == 0).mean():.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x="Book-Rating", data=ratings, color="#4C72B0", ax=axes[0])
axes[0].set_title("All interactions (0 = implicit, no star given)")

explicit_ratings = ratings[ratings["Book-Rating"] > 0]
sns.countplot(x="Book-Rating", data=explicit_ratings, color="#DD8452", ax=axes[1])
axes[1].set_title("Explicit ratings only (1-10)")

plt.tight_layout()
plt.show()


In [ ]:
n_users =  # Write your code here
n_books =  # Write your code here
n_ratings = len(ratings)
sparsity = 1 - n_ratings / (n_users * n_books)

print(f"Unique users:   {n_users:,}")
print(f"Unique books:   {n_books:,}")
print(f"Total ratings:  {n_ratings:,}")
print(f"Matrix sparsity: {sparsity:.4%}  <-- almost every cell of the full user-item matrix is EMPTY")


> 🔑 **Trick #1 — Sparsity is the defining challenge of recommender systems.** A ~99.99% sparse matrix means
> the *average* user has rated a tiny fraction of a percent of the catalog. Any algorithm that needs a dense
> matrix (like naive user-user comparison over the *full* catalog) will be painfully slow and mostly comparing
> empty cells. This is exactly why we filter down to active users/popular items next, and why production
> systems use approximate nearest-neighbor search instead of brute force.


## 5. Data Cleaning & Filtering ✂️

Two cleaning decisions, both standard practice in real recommender pipelines:

1. **Keep explicit ratings only** (`Book-Rating > 0`). A "0" just means the user logged an interaction with no
   opinion attached — mixing it in with a real "I loved this book, 9/10" would badly confuse any model trained
   to predict rating *value*.
2. **Filter to active users and popular books** (at least `MIN_RATINGS` each). This isn't cheating — it's the
   same "min support" trick used everywhere in industry. A user with 1 rating or a book with 1 rating gives a
   model almost nothing to learn from, and keeping them just adds sparsity and compute cost for zero benefit.


In [ ]:
MIN_RATINGS =  # Write your code here  # keep users & books with at least this many explicit ratings

explicit =  # Write your code here

user_counts =  # Write your code here
book_counts =  # Write your code here

active_users = user_counts[user_counts >= MIN_RATINGS].index
popular_books = book_counts[book_counts >= MIN_RATINGS].index

filtered = explicit[explicit["User-ID"].isin(active_users) & explicit["ISBN"].isin(popular_books)].copy()

# keep only books that actually have metadata in Books.csv (a handful of ISBNs in Ratings.csv
# don't -- this is normal real-world messiness, we just don't want to *recommend* an item we
# can't show a title for!)
filtered = filtered[filtered["ISBN"].isin(books["ISBN"])].copy()

print(f"Before filtering: {len(explicit):,} explicit ratings, "
      f"{explicit['User-ID'].nunique():,} users, {explicit['ISBN'].nunique():,} books")
print(f"After filtering:  {len(filtered):,} explicit ratings, "
      f"{filtered['User-ID'].nunique():,} users, {filtered['ISBN'].nunique():,} books")


### One train/test split, used by every model 🎯

Every technique below will be **trained** on the same `train_ratings` and **honestly evaluated** on the same
held-out `test_ratings`. This matters more than it might look: if we built our similarity matrix or SVD factors
using ratings that are also in the test set, we would be "peeking" at the answers before grading ourselves —
exactly the **evaluation leakage** pitfall we warn about in Section 16. Splitting once, up front, and reusing
the split everywhere keeps every technique honest and directly comparable.


In [ ]:
train_ratings, test_ratings = train_test_split(filtered, test_size=0.2, random_state=RANDOM_STATE)
print(f"train_ratings: {len(train_ratings):,} ratings   test_ratings: {len(test_ratings):,} ratings")


## 6. Build the User–Item Rating Matrix

This dense `users x books` matrix (0 = no rating) is the core data structure that user-based, item-based, and
matrix-factorization CF are all built on top of. **Built from `train_ratings` only** — the test ratings stay
completely unseen until Section 14.


In [ ]:
user_item_matrix = train_ratings.pivot_table(index="User-ID", columns="ISBN", values="Book-Rating").fillna(0)
print("User-item matrix shape:", user_item_matrix.shape)

new_sparsity = 1 - (user_item_matrix > 0).sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1])
print(f"Sparsity after filtering: {new_sparsity:.2%}  (much denser, but still mostly empty -- that's normal!)")

title_of = books.drop_duplicates("ISBN").set_index("ISBN")["Book-Title"].to_dict()
author_of = books.drop_duplicates("ISBN").set_index("ISBN")["Book-Author"].to_dict()

user_item_matrix.iloc[:5, :5]


## 7. Technique 1 — Popularity-Based Recommender 🏆

The simplest recommender that exists: recommend whatever is popular and highly rated, to *everyone*. It sounds
too dumb to matter, but it's the backbone of every real system for one crucial reason:

> 🔑 **Trick #2 — Popularity is your cold-start fallback.** A brand-new user has zero history. Every
> personalized algorithm below is undefined for them. Every production recommender therefore falls back to a
> popularity list (often "trending in your country/category") until the user has generated enough history to
> personalize.

**A subtlety:** don't just sort by average rating — a book with a single 10/10 rating would beat a book with
500 ratings averaging 9.2. We use a **Bayesian average** that pulls low-count items toward the global mean,
which is the same trick IMDb and Steam use for their "Top Rated" lists.


In [ ]:
book_stats = train_ratings.groupby("ISBN").agg(
    n_ratings=("Book-Rating", "count"),
    avg_rating=("Book-Rating", "mean"),
).reset_index()

C = book_stats["n_ratings"].quantile(0.90)   # confidence threshold: ~90th percentile of rating counts
m = train_ratings["Book-Rating"].mean()      # global average rating

book_stats["bayesian_score"] = (
    (book_stats["n_ratings"] / (book_stats["n_ratings"] + C)) * book_stats["avg_rating"]
    + (C / (book_stats["n_ratings"] + C)) * m
)

def recommend_popular(user_id=None, k=10):
    # Same top-K list for everyone -- `user_id` is accepted (and ignored) so this function has the
    # same (user_id, k) signature as every personalized recommend_* function below, making it a
    # drop-in cold-start fallback anywhere in this notebook.
    top = book_stats.sort_values("bayesian_score", ascending=False).head(k).copy()
    top["Book-Title"] = top["ISBN"].map(title_of)
    top["Book-Author"] = top["ISBN"].map(author_of)
    return top[["ISBN", "Book-Title", "Book-Author", "n_ratings", "avg_rating", "bayesian_score"]].reset_index(drop=True)

recommend_popular(k=10)


## 8. Technique 2 — User-Based Collaborative Filtering 👥

**Core idea:** *"Find users whose rating patterns look like mine, then recommend what they liked that I
haven't read yet."*

Steps:
1. Compute similarity between every pair of users (cosine similarity on their rating vectors)
2. For a target user, find the `K` most similar users (their "neighborhood")
3. Recommend books the neighborhood rated highly that the target user hasn't rated


In [ ]:
user_similarity =  # Write your code here
user_sim_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)
print("User similarity matrix:", user_sim_df.shape)

def recommend_user_based(user_id, k_neighbors=20, n_recs=10):
    if user_id not in user_sim_df.index:
        return recommend_popular(k=n_recs)  # cold-start fallback (Trick #2 again!)

    # K most similar OTHER users
    neighbors = user_sim_df[user_id].drop(user_id).sort_values(ascending=False).head(k_neighbors)

    already_read = set(user_item_matrix.columns[user_item_matrix.loc[user_id] > 0])

    # weighted average rating from the neighborhood for every book the target user hasn't read
    neighbor_ratings = user_item_matrix.loc[neighbors.index]
    weights = neighbors.values.reshape(-1, 1)
    weighted_scores = (neighbor_ratings.values * weights).sum(axis=0) / (np.abs(weights).sum() + 1e-9)
    scores = pd.Series(weighted_scores, index=user_item_matrix.columns)
    scores = scores.drop(index=[b for b in already_read if b in scores.index])

    top = scores.sort_values(ascending=False).head(n_recs).reset_index()
    top.columns = ["ISBN", "predicted_score"]
    top["Book-Title"] = top["ISBN"].map(title_of)
    top["Book-Author"] = top["ISBN"].map(author_of)
    return top[["Book-Title", "Book-Author", "predicted_score"]]

demo_user = user_item_matrix.index[0]
print(f"User-based recommendations for user {demo_user}:")
recommend_user_based(demo_user)


> 🔑 **Trick #3 — User-based CF doesn't scale.** Computing an `N_users x N_users` similarity matrix is
> `O(N^2)`. With 280,000 real users this is already expensive, and platforms with hundreds of millions of users
> can't afford it at all. Also, individual user taste can be noisy and drifts constantly (new users appear
> every second). This is exactly why most production systems prefer **item-based** CF next, or precompute
> everything offline and refresh on a schedule.


## 9. Technique 3 — Item-Based Collaborative Filtering 📖➡️📖

**Core idea, flipped:** *"Find items that tend to be rated similarly by the same users, then recommend items
similar to what I already liked."* This is exactly Amazon's classic **"Customers who bought this item also
bought"**.

Why this usually wins in production over user-based CF:
* The catalog (items) is usually far more stable than the user base — new users sign up every second, but a
  book's similarity to other books barely changes day to day, so the similarity matrix can be **precomputed
  once and cached**.
* `N_items` is very often much smaller than `N_users` (fewer books than readers), so the similarity matrix is
  cheaper to build and store.


In [ ]:
item_similarity =  # Write your code here
item_sim_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)
print("Item similarity matrix:", item_sim_df.shape)

def similar_books(isbn, n=5):
    if isbn not in item_sim_df.index:
        return pd.DataFrame()
    sims = item_sim_df[isbn].drop(isbn).sort_values(ascending=False)
    out = sims.reset_index()
    out.columns = ["ISBN", "similarity"]
    out["Book-Title"] = out["ISBN"].map(title_of)
    out["Book-Author"] = out["ISBN"].map(author_of)
    # the Book-Crossing catalog has multiple ISBNs for the same title (different print editions) --
    # dedupe so the list isn't just 4 rows of the same book
    out = out.drop_duplicates(subset="Book-Title").head(n)
    return out[["Book-Title", "Book-Author", "similarity"]].reset_index(drop=True)

def recommend_item_based(user_id, k=10):
    # Personalized item-based recommendation: score every candidate book by its total similarity
    # to everything this user has already rated highly.
    if user_id not in user_item_matrix.index:
        return recommend_popular(k=k)
    user_ratings = user_item_matrix.loc[user_id]
    rated = user_ratings[user_ratings > 0]
    if rated.empty:
        return recommend_popular(k=k)
    weights = item_sim_df[rated.index].values @ rated.values
    scores = pd.Series(weights, index=item_sim_df.index)
    scores = scores.drop(index=[b for b in rated.index if b in scores.index])
    top = scores.sort_values(ascending=False).head(k).reset_index()
    top.columns = ["ISBN", "score"]
    top["Book-Title"] = top["ISBN"].map(title_of)
    top["Book-Author"] = top["ISBN"].map(author_of)
    return top[["ISBN", "Book-Title", "Book-Author", "score"]]

# pick a well-known, well-rated book from our filtered catalog to demo with
seed_isbn = book_stats.sort_values("n_ratings", ascending=False).iloc[0]["ISBN"]
print(f"Because you liked: {title_of.get(seed_isbn)!r}\n")
similar_books(seed_isbn, n=5)


`similar_books` answers *"more like this one book."* `recommend_item_based` answers the more useful,
fully personalized question: *"given this user's entire rating history, what should we show them next?"* —
by summing similarity across everything they've rated, weighted by how much they liked each one.


In [ ]:
print(f"Item-based recommendations for user {demo_user}:")
recommend_item_based(demo_user)


## 10. Technique 4 — Model-Based CF: Matrix Factorization (SVD) 🧩

User-based and item-based CF work directly on raw similarity between rows/columns of the matrix. **Matrix
factorization** instead assumes every user and every book can be described by a small number of hidden
**latent factors** (maybe ~20 numbers each) — think loosely of dimensions like "how much romance," "how much
plot complexity," "how literary" — except the model discovers these factors itself; it never labels them for
us.

$$ \hat{R} \approx U \cdot V^T $$

where $U$ is a `(users x k)` matrix and $V$ is a `(items x k)` matrix. Multiplying a user's factor vector by
a book's factor vector predicts that user's rating for that book — including for books the user never
rated!

This exact idea (refined into a very well-tuned form) is what **won the $1,000,000 Netflix Prize** in 2009.
We'll use `TruncatedSVD` from scikit-learn as a fast, teachable version of it.


In [ ]:
N_FACTORS =  # Write your code here

# Mean-centering trick: a raw, un-centered matrix is >99% zeros, so a plain SVD mostly just learns
# "this user rated almost nothing" rather than real taste signal. Instead we subtract each user's
# own average rating first (so the model factorizes *deviations* from personal taste, not raw
# scores), factorize that, then add each user's average back onto the reconstruction.
global_mean = train_ratings["Book-Rating"].mean()
user_means = user_item_matrix.replace(0, np.nan).mean(axis=1).fillna(global_mean)
demeaned_matrix = user_item_matrix.sub(user_means, axis=0)
demeaned_matrix = demeaned_matrix.where(user_item_matrix > 0, 0)  # keep unrated cells at 0 (= "assume average")

svd = TruncatedSVD(n_components=N_FACTORS, random_state=RANDOM_STATE)
user_factors = svd.fit_transform(demeaned_matrix.values)   # (n_users, k)
item_factors = svd.components_.T                             # (n_items, k)

print("User latent factors:", user_factors.shape)
print("Item latent factors:", item_factors.shape)
print(f"Variance explained by {N_FACTORS} latent factors: {svd.explained_variance_ratio_.sum():.1%}")

reconstructed = user_factors @ item_factors.T
predicted_ratings = np.clip(reconstructed + user_means.values.reshape(-1, 1), 1, 10)
predicted_df = pd.DataFrame(predicted_ratings, index=user_item_matrix.index, columns=user_item_matrix.columns)

def recommend_svd(user_id, n_recs=10):
    if user_id not in predicted_df.index:
        return recommend_popular(k=n_recs)
    already_read = set(user_item_matrix.columns[user_item_matrix.loc[user_id] > 0])
    scores = predicted_df.loc[user_id].drop(index=[b for b in already_read if b in predicted_df.columns])
    top = scores.sort_values(ascending=False).head(n_recs).reset_index()
    top.columns = ["ISBN", "predicted_rating"]
    top["Book-Title"] = top["ISBN"].map(title_of)
    top["Book-Author"] = top["ISBN"].map(author_of)
    return top[["ISBN", "Book-Title", "Book-Author", "predicted_rating"]]

recommend_svd(demo_user)


> 🔑 **Trick #4 — Fewer, denser numbers beat more, sparser ones.** Instead of comparing sparse 3,000-dimension
> rating vectors directly, SVD compresses every user and item down to ~20 dense numbers. This is faster,
> generalizes better (it's forced to find real *patterns*, not memorize individual ratings), and is the direct
> conceptual ancestor of today's neural embeddings, which we build next.


## 11. Technique 5 — Content-Based Filtering 🏷️

All four techniques above share one Achilles' heel: **the item cold-start problem.** A book published
yesterday has *zero* ratings, so it can never surface in user-based, item-based, or SVD recommendations — it
literally isn't in the matrix.

**Content-based filtering solves this** by using the book's own metadata (title + author here) instead of
other users' ratings. We turn each book's text into a **TF-IDF vector** and recommend books whose text is
*textually* similar — this works even for a book with zero ratings, the moment it's added to the catalog.


In [ ]:
catalog = books.drop_duplicates("ISBN").copy()
catalog["text"] = (catalog["Book-Title"].fillna("") + " " + catalog["Book-Author"].fillna(""))

tfidf = TfidfVectorizer(stop_words="english", max_features=20000)
tfidf_matrix = tfidf.fit_transform(catalog["text"])
print("TF-IDF matrix:", tfidf_matrix.shape)

catalog = catalog.reset_index(drop=True)
isbn_to_row = pd.Series(catalog.index, index=catalog["ISBN"])

def recommend_content_based(isbn, n=5):
    if isbn not in isbn_to_row.index:
        return pd.DataFrame()
    row = isbn_to_row[isbn]
    sims = cosine_similarity(tfidf_matrix[row], tfidf_matrix).flatten()
    # over-fetch, then dedupe by title -- the catalog has many different ISBNs (print editions)
    # sharing the exact same title + author, which would otherwise flood the top of this list
    top_idx = np.argsort(-sims)[1:n * 8 + 1]
    out = catalog.iloc[top_idx][["Book-Title", "Book-Author"]].copy()
    out["similarity"] = sims[top_idx]
    out = out.drop_duplicates(subset="Book-Title").head(n)
    return out.reset_index(drop=True)

print(f"Because you liked: {title_of.get(seed_isbn)!r}\n")
recommend_content_based(seed_isbn, n=5)


> 🔑 **Trick #5 — Combine, don't choose.** In production, nobody picks *just* collaborative filtering *or*
> content-based. A common pattern: use content-based for genuinely new items until they accumulate ~10-20
> ratings, then let collaborative filtering take over. That's a **hybrid recommender**, and it's the industry
> default.


## 12. Technique 6 — Deep Learning-Based: Neural Collaborative Filtering (NCF) 🧠

SVD predicts a rating as a fixed **dot product** of a user vector and item vector. What if we let a neural
network learn a more flexible, *non-linear* function instead? That's the core idea behind **Neural
Collaborative Filtering** — and the same embedding-based idea, scaled up massively, is what powers
recommendation at YouTube, TikTok, and Instagram.

**Architecture:**
1. An `Embedding` layer turns each `User-ID` into a dense learned vector (just like SVD's latent factors —
   except this time gradient descent learns them, not linear algebra)
2. Another `Embedding` layer does the same for each `ISBN`
3. We concatenate the two vectors and pass them through a small **MLP (multi-layer perceptron)** that learns
   arbitrary interactions between user taste and item characteristics
4. The output is a single predicted rating


In [ ]:
# Reuse the SAME train_ratings / test_ratings split from Section 5 -- same rule as every other
# model in this notebook: train only on train_ratings, never look at test_ratings until evaluation.
user_ids_unique = filtered["User-ID"].unique()
item_ids_unique = filtered["ISBN"].unique()
user2idx = {u: i for i, u in enumerate(user_ids_unique)}
item2idx = {b: i for i, b in enumerate(item_ids_unique)}

train_df = train_ratings.copy()
test_df = test_ratings.copy()
train_df["u"] = train_df["User-ID"].map(user2idx)
train_df["i"] = train_df["ISBN"].map(item2idx)
test_df["u"] = test_df["User-ID"].map(user2idx)
test_df["i"] = test_df["ISBN"].map(item2idx)

n_users_nn = len(user_ids_unique)
n_items_nn = len(item_ids_unique)

print(f"Training on {len(train_df):,} interactions, testing on {len(test_df):,}")
print(f"{n_users_nn:,} users x {n_items_nn:,} items")


In [ ]:
EMBEDDING_DIM = 16

user_input = layers.Input(shape=(1,), name="user")
item_input = layers.Input(shape=(1,), name="item")

user_embedding = layers.Embedding(n_users_nn, EMBEDDING_DIM, name="user_embedding")(user_input)
item_embedding = layers.Embedding(n_items_nn, EMBEDDING_DIM, name="item_embedding")(item_input)

user_vec = layers.Flatten()(user_embedding)
item_vec = layers.Flatten()(item_embedding)

x = layers.Concatenate()([user_vec, item_vec])
x = layers.Dense(32, activation="relu")(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(16, activation="relu")(x)
output = layers.Dense(1, name="predicted_rating")(x)

ncf_model = keras.Model(inputs=[user_input, item_input], outputs=output, name="NeuralCF")
ncf_model.compile(optimizer="adam", loss="mse")
ncf_model.summary()


In [ ]:
history = ncf_model.fit(
    [train_df["u"], train_df["i"]], train_df["Book-Rating"],
    validation_data=([test_df["u"], test_df["i"]], test_df["Book-Rating"]),
    epochs=8, batch_size=256, verbose=2,
)

pd.DataFrame(history.history).plot(figsize=(7, 4), title="Neural CF training curve")
plt.xlabel("epoch"); plt.ylabel("MSE loss")
plt.show()


In [ ]:
ncf_pred = np.clip(ncf_model.predict([test_df["u"], test_df["i"]], verbose=0).flatten(), 1, 10)
ncf_rmse = np.sqrt(mean_squared_error(test_df["Book-Rating"], ncf_pred))
print(f"Neural CF test RMSE: {ncf_rmse:.3f}  (ratings are on a 1-10 scale)")

idx2item = {v: k for k, v in item2idx.items()}

def recommend_ncf(user_id, n_recs=10):
    if user_id not in user2idx:
        return recommend_popular(k=n_recs)
    u_idx = user2idx[user_id]
    all_item_idx = np.arange(n_items_nn)
    u_arr = np.full_like(all_item_idx, u_idx).reshape(-1, 1)
    i_arr = all_item_idx.reshape(-1, 1)
    preds = np.clip(ncf_model.predict([u_arr, i_arr], verbose=0).flatten(), 1, 10)

    already_read_isbns = set(user_item_matrix.columns[user_item_matrix.loc[user_id] > 0]) if user_id in user_item_matrix.index else set()
    scores = pd.Series(preds, index=[idx2item[i] for i in all_item_idx])
    scores = scores.drop(index=[b for b in already_read_isbns if b in scores.index])

    top = scores.sort_values(ascending=False).head(n_recs).reset_index()
    top.columns = ["ISBN", "predicted_rating"]
    top["Book-Title"] = top["ISBN"].map(title_of)
    top["Book-Author"] = top["ISBN"].map(author_of)
    return top[["ISBN", "Book-Title", "Book-Author", "predicted_rating"]]

recommend_ncf(demo_user)


> 🔑 **Trick #6 — Deep learning is not automatically "better."** With only ~60k interactions, SVD and Neural
> CF will land in a similar ballpark, and SVD trains in milliseconds versus seconds for NCF. Deep learning
> pulls ahead when you have (a) **much more data**, (b) **extra features** to fuse in (text, images, session
> sequence, time of day...) that a plain dot product can't use, or (c) a need for very flexible, non-linear
> interactions. Always try the simple linear-algebra baseline (SVD) first — it's a very strong baseline that's
> cheap to build and easy to explain to stakeholders.


## 13. Comparing What We Built So Far 📊

Let's put the two techniques that predict an actual **rating value** (so we can compute RMSE for both)
side by side: classical **SVD** vs. **Neural CF**.


In [ ]:
# SVD's user/item factors only exist for users & books seen in train_ratings, so we can only
# honestly score the subset of test_ratings where both the user and the book were seen in training
# -- this is the SVD model's own version of the cold-start limitation from Section 7/11.
eval_mask = test_ratings["ISBN"].isin(user_item_matrix.columns) & test_ratings["User-ID"].isin(user_item_matrix.index)
svd_eval = test_ratings[eval_mask]
svd_preds = np.array([predicted_df.loc[u, b] for u, b in zip(svd_eval["User-ID"], svd_eval["ISBN"])])
svd_rmse = np.sqrt(mean_squared_error(svd_eval["Book-Rating"], svd_preds))
print(f"SVD could score {len(svd_eval):,} / {len(test_ratings):,} test ratings "
      f"({len(svd_eval)/len(test_ratings):.0%}) -- the rest involve a book/user missing from train_ratings.")

comparison = pd.DataFrame({
    "Model": ["Matrix Factorization (SVD)", "Neural Collaborative Filtering"],
    "Test RMSE (lower better)": [svd_rmse, ncf_rmse],
    "Training time": ["Milliseconds", "Seconds (CPU)"],
    "Handles new items?": ["No", "No"],
    "Handles new users?": ["No", "No"],
})
comparison


## 14. Evaluation Metrics Beyond RMSE 🎯

RMSE measures how close a *predicted rating* is to the *true rating* — but in production, nobody shows a user
a predicted number. They show a **ranked list** of items. What matters is: **of the items we recommended, how
many did the user actually want?** That's a ranking/retrieval problem, evaluated differently:

| Metric | Question it answers |
|---|---|
| **Precision@K** | Of the `K` items we recommended, what fraction did the user actually like? |
| **Recall@K** | Of all the items the user actually liked, what fraction did we manage to surface in our top `K`? |
| **Coverage** | What fraction of the *entire catalog* does the system ever recommend to *anyone*? (Low coverage = we only ever recommend the same popular items — a real business risk!) |

We'll treat any rating ≥ 7 (out of 10) in the *held-out* test set as "the user liked it," and check whether
our recommenders' top-10 lists catch those liked books.


In [ ]:
def precision_recall_at_k(recommend_fn, test_interactions, k=10, like_threshold=7, n_sample_users=150):
    # recommend_fn(user_id, k) must return a DataFrame with an 'ISBN' column -- every recommend_*
    # function we built above already satisfies this contract.
    liked = test_interactions[test_interactions["Book-Rating"] >= like_threshold]
    eval_users = liked["User-ID"].unique()
    rng = np.random.RandomState(RANDOM_STATE)
    if len(eval_users) > n_sample_users:
        eval_users = rng.choice(eval_users, n_sample_users, replace=False)

    precisions, recalls = [], []
    recommended_items_seen = set()

    for u in eval_users:
        true_liked = set(liked.loc[liked["User-ID"] == u, "ISBN"])
        recs = recommend_fn(u, k)
        if recs is None or len(recs) == 0 or "ISBN" not in recs.columns:
            continue
        rec_isbns = set(recs["ISBN"])
        recommended_items_seen |= rec_isbns

        hits = len(rec_isbns & true_liked)
        precisions.append(hits / k)
        recalls.append(hits / len(true_liked) if true_liked else 0)

    coverage = len(recommended_items_seen) / user_item_matrix.shape[1]
    return {
        "Precision@K": float(np.mean(precisions)) if precisions else 0.0,
        "Recall@K": float(np.mean(recalls)) if recalls else 0.0,
        "Coverage": coverage,
        "n_users_evaluated": len(precisions),
    }

results_popular = precision_recall_at_k(recommend_popular, test_ratings, k=10)
results_item_based = precision_recall_at_k(recommend_item_based, test_ratings, k=10)

pd.DataFrame([
    {"Model": "Popularity baseline", **results_popular},
    {"Model": "Item-Based CF", **results_item_based},
])


Notice the trade-off you'll almost always see in this table: the **popularity baseline** often has
*decent* Precision@K (popular books are popular for a reason — lots of people like them!) but usually **much
lower Coverage** — it recommends the same handful of bestsellers to everyone. **Item-based CF** should show
better **Coverage**, because different users get different, personalized lists.

> 🔑 **Trick #7 — Never evaluate a recommender on accuracy alone.** A system that only ever recommends the #1
> global bestseller could still score well on precision while providing zero real personalization and zero
> business differentiation. Always look at coverage/diversity alongside precision & recall.


## 15. A Guided Tour of Modern, Production-Grade Techniques 🚀

Everything above is foundational and still runs in production today — but at YouTube/TikTok/Amazon scale,
a few more ideas get layered on top:

| Technique | Core idea | Used by |
|---|---|---|
| **Two-Tower Retrieval** | Train *two separate* neural networks — one that embeds users, one that embeds items — into the *same* vector space, then use fast approximate-nearest-neighbor search (e.g. FAISS) to retrieve candidates from millions of items in milliseconds | YouTube, most large-scale "candidate generation" stages |
| **Sequential / Session-Based Recommenders** | Instead of "what does this user like in general," model "what will this user click **next**, given the last 10 things they clicked" — uses Transformer/RNN architectures (e.g. SASRec, BERT4Rec) | TikTok's For You feed, Amazon "Buy it again" |
| **Graph Neural Networks (e.g. LightGCN)** | Treat users & items as nodes in a graph connected by interactions, and propagate signal across multiple hops ("my friend's friend liked this") | Pinterest (PinSage), Alibaba |
| **LLM-Based / Conversational Recommenders** | Use a large language model to reason over item descriptions/reviews in natural language, or let users *describe* what they want in plain English | Newer product features (e.g. conversational shopping assistants) |
| **Hybrid / Multi-Stage Pipelines** | Real systems are rarely *one* algorithm: a cheap **candidate generation** stage (e.g. two-tower or item-based CF) narrows millions of items to a few hundred, then a heavier **ranking** model (e.g. a deep neural net with dozens of features) re-scores just those few hundred | Virtually every large-scale recommender in production |

> 💡 **The pattern to remember:** production-scale recommendation is almost always **two-stage**: a fast, wide
> *retrieval* step (get candidates down from millions to hundreds) followed by a slower, precise *ranking* step
> (pick the best few from those hundreds). Everything we built today — popularity, item-based CF, SVD — is a
> perfectly reasonable **retrieval** stage. Neural CF-style models are a reasonable **ranking** stage.


## 16. Practical Tricks & Pitfalls Every Practitioner Should Know ⚠️

| Pitfall | What goes wrong | The fix |
|---|---|---|
| **Cold-start users** | Brand-new user has no history → every personalized model is undefined | Fall back to popularity, or ask onboarding questions ("pick 3 genres you like") |
| **Cold-start items** | Brand-new item has no ratings → collaborative filtering can never surface it | Content-based filtering, or manually boost new items for a trial period |
| **Popularity bias** | The model over-recommends already-popular items, which get more clicks, which reinforces the bias (a feedback loop) | Track & report **coverage/diversity**, not just accuracy; consider explicitly re-ranking for diversity |
| **Implicit vs. explicit feedback** | A "0" rating or a click is *not* the same signal as a 1-star vs. 10-star rating; conflating them corrupts the model | Model them separately, or use implicit-feedback-specific methods (e.g. ALS for implicit feedback) instead of treating every "0" as a bad rating |
| **Negative sampling** | Most datasets only record what users *liked* (positive signal) — an all-positive training set teaches a model to predict "everything is great" | Randomly sample items the user *didn't* interact with as pseudo-negatives during training |
| **Evaluation leakage** | Randomly splitting rows into train/test can let a user's *future* rating leak into training, inflating the reported score | Use a **time-based split**: train on ratings before date X, test on ratings after date X |
| **Scalability at inference** | Brute-force cosine similarity across millions of items is too slow for real-time serving | Approximate nearest-neighbor libraries (FAISS, ScaNN, Annoy) |


## 17. Turning Recommendations into a Business Action 💰

A recommender that only ever gets shown in a Jupyter notebook makes no money. The last, most important step is
connecting a user segment to a concrete action:

| User segment | Which technique applies | Business action |
|---|---|---|
| **Brand-new visitor** (no account, no history) | Popularity-based | Show a "Bestsellers" rail on the homepage — zero personalization needed, zero cost to compute |
| **New signed-up user** (a few ratings) | Content-based, seeded from onboarding picks | Ask 3 favorite genres/authors at signup, recommend by content similarity immediately |
| **Established, active reader** | Item-based CF / SVD / Neural CF | Personalized "Recommended for you" email campaign, ranked by predicted rating |
| **Reader who hasn't returned in 60 days** | Item-based CF, seeded from their *last-read* book | "We picked these because you loved [Book]" win-back email |

**Worked example.** Suppose we send our SVD-based top-10 email to the active users in our filtered dataset,
and even a modest 8% open-and-click-through-to-purchase rate holds, at an average book price of $15:


In [ ]:
n_active_users = user_item_matrix.shape[0]
conversion_rate = 0.08
avg_price = 15

expected_revenue = n_active_users * conversion_rate * avg_price
print(f"Active users targeted: {n_active_users:,}")
print(f"Assumed conversion rate: {conversion_rate:.0%}")
print(f"Average book price: ${avg_price}")
print(f"Expected incremental revenue from ONE personalized email send: ${expected_revenue:,.0f}")


That's the entire economic case for building a recommender in one line: **turning a static catalog into
a personalized nudge, multiplied across your whole user base.**


## 18. Summary & Key Takeaways ✅

1. **Recommender systems personalize an overwhelming catalog** — the same problem search solves for explicit
   queries, recommenders solve for implicit intent (Section 1)
2. The **family tree**: popularity → content-based → collaborative filtering (user/item/model-based) →
   deep learning → hybrid — each solves a weakness of the one before it (Section 2)
3. **Rating matrices are almost always extremely sparse** — this single fact drives most of the field's
   engineering decisions, from filtering to embeddings to approximate search (Sections 4-6)
4. **Popularity-based recommenders are not a toy** — they are the mandatory cold-start fallback in every real
   system (Section 7)
5. **Item-based CF usually beats user-based CF in production** because item similarity is more stable and
   cheaper to precompute (Sections 8-9)
6. **Matrix Factorization (SVD)** compresses sparse ratings into dense latent factors — the technique behind
   the Netflix Prize, and the conceptual ancestor of neural embeddings (Section 10)
7. **Content-based filtering is the fix for the item cold-start problem** that all collaborative methods share
   (Section 11)
8. **Neural Collaborative Filtering** replaces SVD's dot product with a learned, non-linear function — powerful
   with enough data, but not automatically better on a small dataset (Section 12)
9. **Evaluate with Precision@K, Recall@K, and Coverage**, not just RMSE — a model can have great accuracy and
   terrible personalization at the same time (Section 14)
10. **Production systems are two-stage**: fast retrieval (candidate generation) followed by precise ranking,
    often combining several of today's techniques together (Section 15)
11. The most important, most-often-skipped step: **connect a recommendation list to a dollars-and-cents
    business action** (Section 17)


---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 6 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>